# Databricks / PySpark dla Data Analyst/Engineer — Moduł 1: Wprowadzenie do Sparka i Databricks

Ten kurs to naturalna kontynuacja kursów Python, SQL/PostgreSQL i Statystyki. Tam
pracowaliśmy na danych, które mieściły się w pamięci jednego komputera. **Apache Spark**
wchodzi do gry, gdy danych jest tak dużo (albo obliczenia są tak kosztowne), że jeden
komputer już nie wystarcza — i trzeba rozłożyć pracę na wiele maszyn naraz.
**Databricks** to najpopularniejsza platforma do uruchamiania Sparka w chmurze, z
notebookami bardzo podobnymi do Jupytera, którego już znasz.

## Spis treści
1. [Czym jest Apache Spark?](#sec1)
2. [Architektura klastra Sparka](#sec2)
3. [RDD kontra DataFrame](#sec3)
4. [SparkSession i pierwszy DataFrame](#sec4)
5. [Leniwa ewaluacja: transformacje kontra akcje](#sec5)
6. [Databricks Free Edition](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


<a id="sec1"></a>
## 1. Czym jest Apache Spark?

**Apache Spark** to silnik do rozproszonego przetwarzania danych — potrafi rozłożyć jedno
duże zadanie (np. przeliczenie miliarda wierszy) na dziesiątki czy setki komputerów
pracujących równolegle, a potem połączyć wyniki w jedną całość. Kluczowe cechy:

- **Przetwarzanie w pamięci** — Spark trzyma dane robocze w RAM-ie między krokami
  obliczeń, zamiast zapisywać je na dysk po każdym etapie (jak robił to starszy
  MapReduce z Hadoopa) — stąd nazwa "spark" (iskra) i duży skok wydajności.
- **Wielojęzyczność** — ten sam silnik obsługuje Python (PySpark, ten kurs), Scalę,
  Javę, R i SQL.
- **Jeden silnik, wiele zastosowań** — przetwarzanie wsadowe (batch), SQL, uczenie
  maszynowe (MLlib), przetwarzanie strumieniowe (Structured Streaming) i grafy — wszystko
  w jednym frameworku.

Wszystko, czego nauczyliśmy się o SQL-u (Moduły z kursu SQL) i o pracy z DataFrame'ami
(`pandas` z kursu Pythona) przenosi się tutaj niemal jeden do jednego — PySpark celowo
naśladuje oba te API.

> ⚠️ **Spark to nie zamiennik pandas/SQL na małych danych**
>
> Jeśli dane mieszczą się wygodnie w pamięci jednego komputera (miliony, a nie miliardy wierszy), zwykły `pandas` albo PostgreSQL z poprzednich kursów są zwykle SZYBSZE i prostsze — uruchomienie Sparka ma swój narzut (uruchomienie klastra, planowanie zadań). Spark błyszczy dopiero, gdy dane naprawdę nie mieszczą się na jednej maszynie, albo gdy obliczenia trzeba rozłożyć na wiele maszyn dla szybkości.

<a id="sec2"></a>
## 2. Architektura klastra Sparka

Aplikacja Sparka działa na **klastrze** złożonym z kilku ról:

- **Driver** — proces "dyrygenta": tu żyje Twój kod (notebook), tu budowany jest plan
  wykonania, tu trafiają zebrane wyniki (np. po `.collect()`).
- **Executory** (*executors*) — procesy robocze na osobnych maszynach, które faktycznie
  wykonują pracę (czytają dane, filtrują, agregują) na swoim fragmencie danych.
- **Menedżer klastra** (*cluster manager*) — przydziela zasoby (ile executorów, ile
  pamięci/CPU) dla Twojej aplikacji. W Databricks tym menedżerem zajmuje się sama
  platforma — nie musisz go konfigurować ręcznie.

Dane są dzielone na **partycje** i rozrzucane między executory — każdy executor
przetwarza swoją partycję **równolegle** z innymi. Do tego, jak dobrze te partycje są
dobrane, wrócimy w Module 9 (optymalizacja).

> ⚡ **Databricks Free Edition = klaster "serverless"**
>
> W darmowej wersji Databricks nie musisz w ogóle myśleć o menedżerze klastra ani liczbie maszyn — dostajesz gotowe obliczenia "serverless" (bez zarządzania infrastrukturą), z rozsądnymi limitami dla nauki i eksperymentów. To, co widzisz w notebooku, działa tak samo niezależnie od tego, ile maszyn faktycznie pracuje w tle.

<a id="sec3"></a>
## 3. RDD kontra DataFrame

Pierwotnym, najniższego poziomu API Sparka jest **RDD** (*Resilient Distributed
Dataset*) — rozproszona kolekcja obiektów Pythona, bardzo elastyczna, ale bez żadnej
wiedzy o strukturze danych (kolumnach, typach), więc Spark nie mógł jej dobrze
optymalizować. **DataFrame** (dodany później) to RDD ze zdefiniowanym schematem
(kolumny + typy) — dzięki temu Spark wie dokładnie, jakie dane przetwarza, i może
zastosować optymalizator zapytań (**Catalyst**) oraz szybszy silnik wykonawczy
(**Tungsten**), podobnie jak silnik bazy danych optymalizuje plan zapytania SQL.

**W tym kursie pracujemy niemal wyłącznie na DataFrame'ach** — to standard we
współczesnym Sparku i to, czego używa się w Databricks na co dzień. RDD wspominamy
tylko po to, żebyś rozumiał, co kryje się pod spodem.

<a id="sec4"></a>
## 4. SparkSession i pierwszy DataFrame

**`SparkSession`** to punkt wejścia do wszystkiego w PySparku — przez nią tworzysz
DataFrame'y, uruchamiasz zapytania SQL, czytasz i zapisujesz pliki. Od razu wczytajmy też
nasz wspólny zestaw danych na resztę kursu — dane sklepu internetowego, dokładnie ten sam
schemat co w kursie SQL (`klienci`, `produkty`, `pracownicy`, `zamowienia`,
`pozycje_zamowien`), ale teraz jako DataFrame'y Sparka.

Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("kurs_spark").getOrCreate()
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


> 🧱 **W Databricks `spark` już istnieje**
>
> W notebooku Databricks zmienna `spark` jest automatycznie gotowa do użycia — nie musisz w ogóle wywoływać `SparkSession.builder...`. Kod powyżej działa też tam (po prostu dostanie już istniejącą sesję), więc możesz go bezpiecznie zostawić w każdym notebooku, żeby działał identycznie lokalnie i w Databricks.

In [ ]:
klienci.printSchema()


In [ ]:
print(f"Liczba klientow: {klienci.count()}")
print(f"Liczba zamowien: {zamowienia.count()}")
print(f"Liczba pozycji zamowien: {pozycje_zamowien.count()}")


<a id="sec5"></a>
## 5. Leniwa ewaluacja: transformacje kontra akcje

To najważniejsza koncepcja mentalna w Sparku, inna niż w `pandas`. Operacje na
DataFrame'ach dzielą się na dwa rodzaje:

- **Transformacje** (*transformations*) — np. `select`, `filter`, `withColumn`,
  `groupBy`. NIE wykonują się od razu — Spark tylko zapisuje sobie "plan", co ma zrobić
  (leniwa ewaluacja, *lazy evaluation*).
- **Akcje** (*actions*) — np. `show()`, `count()`, `collect()`, zapis do pliku. Dopiero
  akcja każe Sparkowi FAKTYCZNIE wykonać cały nagromadzony plan transformacji.

Dzięki temu Spark może przeanalizować CAŁY łańcuch transformacji naraz i zoptymalizować
go (np. połączyć kilka filtrów, pominąć niepotrzebne kolumny), zamiast wykonywać każdy
krok osobno tak, jak napisałeś.

In [ ]:
# Transformacja -- nic sie jeszcze nie liczy, Spark tylko zapisuje plan
plan = klienci.filter(klienci.segment == "Premium").select("klient_id", "miasto")
print(type(plan))


In [ ]:
# Dopiero akcja uruchamia faktyczne obliczenia
plan.show(5)


> ⚡ **explain() pokazuje plan, zanim cokolwiek się wykona**
>
> Metoda `.explain()` pokazuje plan wykonania transformacji BEZ ich uruchamiania -- wrócimy do czytania takich planów szczegółowo w Module 9. Spróbuj już teraz: `plan.explain()`.

<a id="sec6"></a>
## 6. Databricks Free Edition

**Databricks** to firma i platforma stworzona przez twórców Apache Spark — najpopularniejszy
sposób na uruchamianie Sparka w chmurze bez samodzielnego zarządzania klastrami.
**Databricks Free Edition** (następca dawnego "Community Edition") to darmowa wersja do
nauki:

- Zakładasz konto samym e-mailem — **nie potrzebujesz własnego konta w chmurze**
  (AWS/Azure/GCP), Databricks udostępnia gotową, darmową infrastrukturę.
- Dostajesz obliczenia typu **"serverless"** (bez ręcznego konfigurowania klastra) oraz
  domyślne miejsce na dane, gotowe od razu po zalogowaniu.
- Do dyspozycji: notebooki (bardzo podobne do tych, na których pracowaliśmy cały ten
  kurs), podstawy Unity Catalog, tworzenie prostych pulpitów (dashboardów) i workflow
  do planowania zadań.
- Ma rozsądne, "fair use" limity wielkości obliczeń — to środowisko do nauki i
  eksperymentów, nie do produkcyjnych obciążeń, ale jest deklarowane jako trwale darmowe
  (bez daty wygaśnięcia).

> 🧱 **Ten notebook a prawdziwy Databricks**
>
> Kod w tym kursie napisaliśmy tak, żeby działał identycznie w zwykłym środowisku Jupyter (jak dotychczas) ORAZ w Databricks Free Edition -- w większości modułów wystarczy wgrać notebook (`.ipynb`) do swojego workspace'u i uruchomić komórki. Od Modułu 10 (Delta Lake) niektóre fragmenty kodu wymagają PRAWDZIWEGO środowiska Databricks (albo lokalnego pobrania dodatkowych bibliotek z internetu) -- oznaczymy je wyraźnie jako "do uruchomienia w Databricks".

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- czym jest Apache Spark i kiedy warto go używać (a kiedy NIE),
- architekturę klastra: driver, executory, menedżer klastra, partycje,
- różnicę między RDD a DataFrame i dlaczego DataFrame jest dziś standardem,
- `SparkSession` jako punkt wejścia do PySparka,
- leniwą ewaluację: transformacje budują plan, akcje go wykonują,
- czym jest Databricks Free Edition i jak z niego skorzystać.

### 📝 Ćwiczenie 1: Transformacja czy akcja?

Dla każdej z poniższych operacji na DataFrame napisz w komentarzu, czy to transformacja czy akcja: `filter()`, `count()`, `select()`, `show()`, `groupBy()`, `collect()`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
# filter()   -- transformacja
# count()    -- akcja
# select()   -- transformacja
# show()     -- akcja
# groupBy()  -- transformacja (samo groupBy jeszcze nic nie liczy, dopiero .agg()+akcja)
# collect()  -- akcja
print("Zobacz komentarze powyzej.")
```

</details>

### 📝 Ćwiczenie 2: Schemat i liczba wierszy

Wypisz schemat (`printSchema()`) oraz liczbę wierszy (`count()`) dla DataFrame'u `produkty`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
produkty.printSchema()
print(f"Liczba produktow: {produkty.count()}")
```

</details>

### 📝 Ćwiczenie 3: Zbuduj plan, ale go nie uruchamiaj

Zbuduj (bez wywoływania żadnej akcji!) transformację, która filtruje `zamowienia` do statusu `'Anulowane'` i wybiera tylko kolumny `zamowienie_id` i `klient_id`. Sprawdź typ zmiennej, w której przechowałeś ten plan.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
plan_anulowane = zamowienia.filter(zamowienia.status == "Anulowane").select("zamowienie_id", "klient_id")
print(type(plan_anulowane))
```

</details>

### 📝 Ćwiczenie 4: Teraz uruchom ten plan

Wywołaj akcję `show()` na planie z ćwiczenia 3, żeby zobaczyć pierwsze 10 anulowanych zamówień.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
plan_anulowane.show(10)
```

</details>

> 🔥 **Wyzwanie: policz anulowane zamówienia bez akcji pośrednich**
>
> Zbuduj JEDEN łańcuch transformacji (bez wywoływania `show()`/`count()` po drodze), który: (1) łączy `zamowienia` z `klienci` (poznamy `join` szczegółowo w Module 4, ale spróbuj już teraz -- `zamowienia.join(klienci, "klient_id")`), (2) filtruje do statusu `'Anulowane'`, (3) grupuje po `segment` i liczy wystąpienia. Wywołaj `.explain()` na tym łańcuchu PRZED jakąkolwiek akcją, a dopiero potem `.show()`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql import functions as F

plan_wyzwanie = (
    zamowienia.join(klienci, "klient_id")
    .filter(zamowienia.status == "Anulowane")
    .groupBy("segment")
    .agg(F.count("*").alias("liczba_anulowanych"))
)
plan_wyzwanie.explain()
plan_wyzwanie.show()
```

Plan pokazany przez explain() laczy filtrowanie, join i agregacje w jeden zoptymalizowany graf wykonania -- to wlasnie robi za Ciebie Catalyst, o ktorym wspomnielismy w sekcji 3.

</details>

## Co dalej?

W **Module 2** zagłębimy się w DataFrame API: jak tworzyć, filtrować, wybierać kolumny i
dodawać nowe — fundament, na którym oprzemy wszystkie kolejne moduły.